In [0]:

from datetime import datetime

modo = "historico" # "automatico"


if modo == "automatico":
  periodo = datetime.now().strftime("%m-%Y")
else :
  periodo = None


In [0]:

## credenciales

jdbc_url = f"jdbc:sqlserver://{host}:{port};databaseName={db}"

In [0]:
tb_full = [
    "TB_CLIENTES_CORE",
    "TB_PRODUCTOS_CAT",
    "TB_SUCURSALES_RED"
]

tb_incremental = {
    "TB_COMISIONES_LOG" : "fec_cobro",
    "TB_MOV_FINANCIEROS" : "fec_mov",
    "TB_OBLIGACIONES" : "fec_desembolso"
}

In [0]:
# df = df \
#     .withColumn("_fecha_extraccion", F.current_timestamp()) \
#     .withColumn("_fuente", F.lit("dbo.TB_CLIENTES_CORE"))

In [0]:
from pyspark.sql import functions as F

In [0]:
bronce_path = "abfss://bronze@stdataknowdeveastus001.dfs.core.windows.net/"

def extract_full(list_tb):
    for i in list_tb:
        df = spark.read.format("jdbc") \
            .option("url",      jdbc_url) \
            .option("dbtable",  "dbo."+i) \
            .option("user",     user) \
            .option("password", password) \
            .option("driver",   driver) \
            .load()

        df = df \
            .withColumn("_fecha_extraccion", F.current_timestamp()) \
            .withColumn("_fuente", F.lit(i))

        df.write.mode("overwrite").format("parquet").save(bronce_path + i)

extract_full(tb_full)



In [0]:
from pyspark.sql.functions import col, date_format
import pyspark.sql.functions as F

def extract_incremental(dict_tb, periodo):

    for key, value in dict_tb.items():

        if periodo is None:

            check_periodo = (
                spark.read.format("jdbc")
                .option("url", jdbc_url)
                .option("dbtable", "dbo."+key)
                .option("user", user)
                .option("password", password)
                .option("driver", driver)
                .load()
                .select(
                    date_format(col(value), "MM-yyyy").alias("periodo")  
                )
                .distinct()
                .collect()
            )

            if len(check_periodo) > 0:
                for row in check_periodo:
                    periodo = row["periodo"]
                    query = (
                        f"(SELECT * FROM {"dbo."+key} WHERE FORMAT({value}, 'MM-yyyy') = '{periodo}') AS subquery"
                    )

                    df = (
                        spark.read.format("jdbc")
                        .option("url", jdbc_url)
                        .option("dbtable", query)
                        .option("user", user)
                        .option("password", password)
                        .option("driver", driver)
                        .load()
                    )

                    df = (
                        df
                        .withColumn("_fecha_extraccion", F.current_timestamp())
                        .withColumn("_fuente", F.lit(key))
                    )

                    df.write.mode("overwrite").format("parquet").save(bronce_path + "/" + key + "/" + periodo)
            else:
                print("No hay registros nuevos")

        else:
            query = (
                f"(SELECT * FROM {"dbo."+key} WHERE FORMAT({value}, 'MM-yyyy') = '{periodo}') AS subquery" 
            )

            df = (
                spark.read.format("jdbc")
                .option("url", jdbc_url)
                .option("dbtable", query)
                .option("user", user)
                .option("password", password)
                .option("driver", driver)
                .load()
            )

            df = (
                df
                .withColumn("_fecha_extraccion", F.current_timestamp())
                .withColumn("_fuente", F.lit(key))
            )

            df.write.mode("overwrite").format("parquet").save(bronce_path + "/" + key + "/" + periodo)


extract_incremental(tb_incremental, periodo)